# **__________   KAYAK PROJECT   ___________**

**The goal of this project is to identify the possible destinations for holidays using APIs and webscrapping to find the best hotels in the cities where the weather forecast suits our wishes best.**

## **I. Identification of the possible destinations according to the weather forecast**

Here our goal will be to identify the 5 best cities for us according to the weather forecast for the 7 following days among a list of 35 most touristic cities in France. The weather API we will use (OpenWeatherMap) needs to be provided with longitude and latitude of each city in order to give us the weather forecast. So we will start to gather these information with another API called Nominatum.

### **1. List of the 35 most touristic cities in France**

One Week In.com identified the top-35 cities to visit in France. We store them into the variable `list_35_cities`.

In [7]:
list_35_cities=["Mont Saint Michel",
"St Malo",
"Bayeux",
"Le Havre",
"Rouen",
"Paris",
"Amiens",
"Lille",
"Strasbourg",
"Chateau du Haut Koenigsbourg",
"Colmar",
"Eguisheim",
"Besancon",
"Dijon",
"Annecy",
"Grenoble",
"Lyon",
"Gorges du Verdon",
"Bormes les Mimosas",
"Cassis",
"Marseille",
"Aix en Provence",
"Avignon",
"Uzes",
"Nimes",
"Aigues Mortes",
"Saintes Maries de la mer",
"Collioure",
"Carcassonne",
"Ariege",
"Toulouse",
"Montauban",
"Biarritz",
"Bayonne",
"La Rochelle"]

### **2. Coordinates of every cities**

We are going to use the information of the **API Nominatim** to get the coordinates of each of the top-35 best cities in France

**2.0. Modules import**

In [8]:
import requests
import pandas as pd
import time
import os

**2.1. API request test on one city : Paris**

To identify the correct parameters for our API request, we can first try a request on a single city and then test it in a loop for all the cities in our list. Let's take Paris for this first exemple.

_2.1.1. Identification of the parameters for the API request_

In the API documentation, we can read that :
- there are different ways to find information, for instance, we can find the location of a city by giving it's name with the "/search" element, or we can do the opposite with "/reverse". Here we need to use the "/search" option.
- the endpoint for a request is "https://nominatim.openstreetmap.org/search?<params>"
- we don't need and authentification
- the parameters we need are `q` for "query", it means the city, the `format` (here we want a json format) and the `addressedetail` to breakdown the adress into elements.

_2.1.2. Creation of the request_

In [9]:

# ------------------------------------- Identify the base URL --------------------------------------

url_base="https://nominatim.openstreetmap.org/search"


# -------------------------- Headers (required to avoid 403 error) ---------------------------------

headers={
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/118.0"
}


# ----------------------------------------- Parameters ---------------------------------------------

params={
    "q":"paris", # In this "query" parameter we select Paris for the fist test 
    "format":"json",
    "addressdetails":"1", # When set to 1, include a breakdown of the address into elements. 
    "limit":"1" # We want only one element per city
}


# --------------------------------------- GET request ----------------------------------------------

response=requests.get(url=url_base,headers=headers,params=params)

# Status analyse
print("Status code:", response.status_code)


# --------------------------------------- JSON display ---------------------------------------------

if response.status_code == 200:
    data_paris_coord = response.json()
    print(data_paris_coord)
else:
    print("Error:", response.text)


# ------------------------------- Get the coordinates of the city ----------------------------------

city_paris="Paris"
lat_paris=data_paris_coord[0]["lat"]
lon_paris=data_paris_coord[0]["lon"]
print(f"\nCoordinates of {city_paris} : (Lat : {lat_paris}, Lon : {lon_paris})")


#--------------------------------- Load the data into a dataframe ---------------------------------

# Creation of a dictionary that represents the row of the DataFrame
dict_paris_coord={
    "City" : city_paris,
    "Latitude" : lat_paris,
    "Longitude" : lon_paris
}

# Creation of the dataframe
df_paris_coord=pd.DataFrame([dict_paris_coord])
display(df_paris_coord)

Status code: 200
[{'place_id': 89081766, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'relation', 'osm_id': 7444, 'lat': '48.8588897', 'lon': '2.3200410', 'class': 'boundary', 'type': 'administrative', 'place_rank': 15, 'importance': 0.897098092136026, 'addresstype': 'suburb', 'name': 'Paris', 'display_name': 'Paris, Île-de-France, France métropolitaine, France', 'address': {'suburb': 'Paris', 'city_district': 'Paris', 'city': 'Paris', 'ISO3166-2-lvl6': 'FR-75C', 'state': 'Île-de-France', 'ISO3166-2-lvl4': 'FR-IDF', 'region': 'France métropolitaine', 'country': 'France', 'country_code': 'fr'}, 'boundingbox': ['48.8155755', '48.9021560', '2.2241220', '2.4697602']}]

Coordinates of Paris : (Lat : 48.8588897, Lon : 2.3200410)


,City,Latitude,Longitude
0,Paris,48.8588897,2.3200410


**2.2. API request for all the cities**

In [10]:

# ------------------------------------- Identify the base URL --------------------------------------

url_base="https://nominatim.openstreetmap.org/search"


# -------------------------- Headers (required to avoid 403 error) ---------------------------------

headers={
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/118.0"
}

# ------------------------------- Loop on the cities in the querry ---------------------------------

list_dict_coord_35_cities=[] # List that will be appended with one dictionary per city

for city in list_35_cities:
    params={
        "q":city, # we loop on the cities of our list 
        "format":"json",
        "addressdetails":"1",
        "limit":"1" 
    }
    response=requests.get(url=url_base,headers=headers,params=params)
    time.sleep(1)

    if response.status_code == 200:
        data = response.json()
    else:
        print("Error:", response.text)


    # Creation of one dictionary per city
    dict_coordinates_35_cities={
        "City" : city,
        "Latitude" : data[0]["lat"],
        "Longitude" : data[0]["lon"]
    }

    # Append each dictionary in the list of dictionaries
    list_dict_coord_35_cities.append(dict_coordinates_35_cities)

**2.3. Creation of a dataframe and export of the data**

In [11]:
# Creation of the dataframe with all the cities' information
df_coord_35_cities=pd.DataFrame(list_dict_coord_35_cities)
display(df_coord_35_cities)

# Export of the dataframe as a CSV file
os.makedirs("DATA/bookinghotels", exist_ok=True)
df_coord_35_cities.to_csv("DATA/bookinghotels/coordinates_35_cities_csv.csv", index=False)

,City,Latitude,Longitude
0,Mont Saint Michel,48.6359541,-1.5114600
1,St Malo,49.3146950,-96.9538228
2,Bayeux,49.2764624,-0.7024738
3,Le Havre,49.4938975,0.1079732
4,Rouen,49.4404591,1.0939658
5,Paris,48.8588897,2.3200410
6,Amiens,49.8941708,2.2956951
7,Lille,50.6365654,3.0635282
8,Strasbourg,48.5846140,7.7507127
9,Chateau du Haut Koenigsbourg,48.2493820,7.3439412


### **3. Weather forecast for every city**

Now that we have the coordinates of each city, we can access the weather forecast **API Openweathermap** to get the weather forecast for the next 7 days and for each of the cities in our list. First we need to register on the website to get our API key.

**3.0. Modules import**

In [12]:
from dotenv import load_dotenv


**3.1. API request test on one city : Paris**

_3.1.1. Identification of the parameters for the API request_

The One Call API 2.5 provides the following weather data for any geographical coordinates (longitude/latitude) that we will provide :

- Current weather
- Minute forecast for 1 hour
- Hourly forecast for 48 hours
- Daily forecast for 7 days
- National weather alerts
- Historical weather data for the previous 5 days
- Current and forecast weat

=> We are interested in the Daily forecast for 7 days so we will select the parameter `daily` by excluding the list `current`,`minutely`,`hourly`,`alerts` in the "exclude" parameter

For the units, we will select the parameter `metric` in order to have the temperature in Celsius and the wind speed in metre/sec (otherwise the default temperature is in Kelvin).

Finally, we will have to use an API key so we will store it in our local environment and use the dotenv library to access it.

_3.1.2. Creation of the request_

In [13]:
!pip install python-dotenv

In [14]:
#------------------------------- Load variables from .env file -----------------------------------

load_dotenv()

# -------------------------- Access the key securely from the environment -------------------------

API_KEY = os.getenv("API_KEY")

# -------------------------------------------- API call -------------------------------------------

url_base="https://api.openweathermap.org/data/2.5/forecast"

params = {
    "lat":lon_paris,
    "lon": lat_paris,
    "units":"metric",
    "exclude":"current,minutely,hourly,alerts",
    "appid":API_KEY,
}

response = requests.get(url_base,params=params)
print(response.url)

# -------------------------------------- Response analysis ----------------------------------------

if response.status_code == 200:
    json_paris_weather = response.json()
    print(json_paris_weather)
else:
    print("Error:", response.text)

https://api.openweathermap.org/data/2.5/forecast?lat=2.3200410&lon=48.8588897&units=metric&exclude=current%2Cminutely%2Chourly%2Calerts&appid=28089930b22b3f568e2d5c225cd39da2
{'cod': '200', 'message': 0, 'cnt': 40, 'list': [{'dt': 1777129200, 'main': {'temp': 29, 'feels_like': 32.73, 'temp_min': 28.7, 'temp_max': 29, 'pressure': 1008, 'sea_level': 1008, 'grnd_level': 1009, 'humidity': 70, 'temp_kf': 0.3}, 'weather': [{'id': 500, 'main': 'Rain', 'description': 'light rain', 'icon': '10n'}], 'clouds': {'all': 87}, 'wind': {'speed': 3.15, 'deg': 29, 'gust': 3.37}, 'visibility': 10000, 'pop': 0.57, 'rain': {'3h': 0.93}, 'sys': {'pod': 'n'}, 'dt_txt': '2026-04-25 15:00:00'}, {'dt': 1777140000, 'main': {'temp': 28.98, 'feels_like': 32.87, 'temp_min': 28.9, 'temp_max': 28.98, 'pressure': 1010, 'sea_level': 1010, 'grnd_level': 1011, 'humidity': 71, 'temp_kf': 0.08}, 'weather': [{'id': 500, 'main': 'Rain', 'description': 'light rain', 'icon': '10n'}], 'clouds': {'all': 94}, 'wind': {'speed': 2.

In [15]:
#---------------------------------- Analysis of the response and the nested dictionaries --------------------------
import json

json_paris_weather=json_paris_weather['list'] # We keep only the list of dictionaries that we need

print(json.dumps(json_paris_weather, indent=4)) # Formatted format for a better readability


[
    {
        "dt": 1777129200,
        "main": {
            "temp": 29,
            "feels_like": 32.73,
            "temp_min": 28.7,
            "temp_max": 29,
            "pressure": 1008,
            "sea_level": 1008,
            "grnd_level": 1009,
            "humidity": 70,
            "temp_kf": 0.3
        },
        "weather": [
            {
                "id": 500,
                "main": "Rain",
                "description": "light rain",
                "icon": "10n"
            }
        ],
        "clouds": {
            "all": 87
        },
        "wind": {
            "speed": 3.15,
            "deg": 29,
            "gust": 3.37
        },
        "visibility": 10000,
        "pop": 0.57,
        "rain": {
            "3h": 0.93
        },
        "sys": {
            "pod": "n"
        },
        "dt_txt": "2026-04-25 15:00:00"
    },
    {
        "dt": 1777140000,
        "main": {
            "temp": 28.98,
            "feels_like": 32.87,
            "

In [16]:
#--------------------------------- Load the data into a dataframe ---------------------------------

# Our Json response contains nested dictionaries so we will use json_normalize to flatten it in order 
# to use it to create a dataframe.

RECORD_PATH = "weather" #record_path`: The key inside each dictionary that points to the LIST of items we 
#want to flatten.


META_FIELDS = [
        'dt',
        ["main","temp"],
        ["main","feels_like"],
        ["main","temp_min"],
        ["main","temp_max"],
        ["main","pressure"],
        ["main","sea_level"],
        ["main","grnd_level"],
        ["main","humidity"],
        ["main","temp_kf"],
        ["clouds","all"],
        ["wind","speed"],
        ["wind","deg"],
        ["wind","gust"],
        "visibility",
        "pop",
        ["sys","pop"],
        "dt_txt"
        ] # `meta`: A list of keys from the *parent* dictionary that 
#we want to keep and repeat for each flattened record.

json_normalized_paris_weather = pd.json_normalize(
data=json_paris_weather,
record_path=RECORD_PATH,
meta=META_FIELDS,
errors='ignore'  # This is important for handling missing/empty lists
)

# Creation of the dataframe
display(json_normalized_paris_weather) # We don't do pd.datatframe() because the normalized json is already a dataframe, we just need to display it



,id,main,description,icon,dt,main.temp,main.feels_like,main.temp_min,main.temp_max,main.pressure,...,main.humidity,main.temp_kf,clouds.all,wind.speed,wind.deg,wind.gust,visibility,pop,sys.pop,dt_txt
0,500,Rain,light rain,10n,1777129200,29,32.73,28.7,29,1008,...,70,0.3,87,3.15,29,3.37,10000,0.57,NaN,2026-04-25 15:00:00
1,500,Rain,light rain,10n,1777140000,28.98,32.87,28.9,28.98,1010,...,71,0.08,94,2.72,49,2.98,10000,0.54,NaN,2026-04-25 18:00:00
2,500,Rain,light rain,10n,1777150800,28.54,32.41,28.54,28.54,1010,...,74,0,100,2.3,39,2.61,10000,0.63,NaN,2026-04-25 21:00:00
3,500,Rain,light rain,10n,1777161600,28.51,32.18,28.51,28.51,1008,...,73,0,100,2.67,17,2.99,10000,0.7,NaN,2026-04-26 00:00:00
4,500,Rain,light rain,10d,1777172400,28.62,32.08,28.62,28.62,1010,...,71,0,100,2.68,11,2.59,10000,0.75,NaN,2026-04-26 03:00:00
5,500,Rain,light rain,10d,1777183200,28.54,31.91,28.54,28.54,1012,...,71,0,100,1.3,359,1.23,10000,0.72,NaN,2026-04-26 06:00:00
6,500,Rain,light rain,10d,1777194000,28.33,31.62,28.33,28.33,1010,...,72,0,100,1.02,320,1.08,10000,0.6,NaN,2026-04-26 09:00:00
7,500,Rain,light rain,10d,1777204800,28.21,31.52,28.21,28.21,1009,...,73,0,100,2.22,324,2.54,10000,0.74,NaN,2026-04-26 12:00:00
8,500,Rain,light rain,10n,1777215600,28.6,31.87,28.6,28.6,1010,...,70,0,100,2.42,344,2.56,10000,0.72,NaN,2026-04-26 15:00:00
9,500,Rain,light rain,10n,1777226400,28.7,32.08,28.7,28.7,1011,...,70,0,100,1.71,49,2.01,10000,0.61,NaN,2026-04-26 18:00:00


**List of variable in the json response and the data frame that we wish to keep (checked variables):**

Variables not nested:
- [ ] **`dt`** : Time of data forecasted, unix, UTC
- [x] **`visibility`**: Average visibility (meter) 
- [x] **`pop`**: Probability of precipitation (0 to 1 where 0=0% and 1=100%) 
- [x] **`dt_txt`** #  Time of data forecasted, ISO, UTC 


Variables in **`main`** : 
- [x] **`temp`** : Temperature (°C) 
- [x] **`feels_like`**: Human perception of the temperature (°C) 
- [x] **`temp_min`**: Minimum temperature at the moment of calculation (°C) . This is minimal forecasted temperature (within large megalopolises and urban areas), use this parameter optionally.
- [x] **`temp_max`**: Maximum temperature at the moment of calculation (°C).  This is minimal forecasted temperature (within large megalopolises and urban areas), use this parameter optionally.
- [ ] **`pressure`**: Atmospheric pressure on the sea level (hPa)
- [ ] **`sea_level`**: Atmospheric pressure on the sea level (hPa)
- [ ] **`grnd_level`**: Atmospheric pressure on the ground level (hPa)
- [x] **`humidity`**: Humidity % 
- [ ] **`temp_kf`**: Internal parameter

Variables in **`weather`** : 
- [ ] **`id`**: Weather condition id
- [ ] **`main`**: Group of weather parameters (Rain, Snow, Clouds etc.)
- [x] **`description`**: Weather condition within the group 
- [ ] **`icon`**: Weather icon id
  
Variable in **`cloud`** : 
- [x] **`all`**: Cloudiness, %

Variables in **`wind`**:
- [x] **`speed`**: Wind speed (meter/sec) 
- [ ] **`deg`**: Wind direction (degree)
- [x] **`gust`**: Wind gust (meter/sec) 

Variable in **`sys`** : 
- [x] **`pop`**: Part of the day (n=night, d=day) 


In [17]:
#----------------------------------- Selection of columns of interest -------------------------------------

columns_to_keep=['dt_txt','main.temp','main.feels_like','main.temp_min','main.temp_max','main.humidity','description','clouds.all','pop','sys.pop','wind.speed','wind.gust','visibility']
df_weather_paris= json_normalized_paris_weather[columns_to_keep].copy()

display(df_weather_paris)


,dt_txt,main.temp,main.feels_like,main.temp_min,main.temp_max,main.humidity,description,clouds.all,pop,sys.pop,wind.speed,wind.gust,visibility
0,2026-04-25 15:00:00,29,32.73,28.7,29,70,light rain,87,0.57,NaN,3.15,3.37,10000
1,2026-04-25 18:00:00,28.98,32.87,28.9,28.98,71,light rain,94,0.54,NaN,2.72,2.98,10000
2,2026-04-25 21:00:00,28.54,32.41,28.54,28.54,74,light rain,100,0.63,NaN,2.3,2.61,10000
3,2026-04-26 00:00:00,28.51,32.18,28.51,28.51,73,light rain,100,0.7,NaN,2.67,2.99,10000
4,2026-04-26 03:00:00,28.62,32.08,28.62,28.62,71,light rain,100,0.75,NaN,2.68,2.59,10000
5,2026-04-26 06:00:00,28.54,31.91,28.54,28.54,71,light rain,100,0.72,NaN,1.3,1.23,10000
6,2026-04-26 09:00:00,28.33,31.62,28.33,28.33,72,light rain,100,0.6,NaN,1.02,1.08,10000
7,2026-04-26 12:00:00,28.21,31.52,28.21,28.21,73,light rain,100,0.74,NaN,2.22,2.54,10000
8,2026-04-26 15:00:00,28.6,31.87,28.6,28.6,70,light rain,100,0.72,NaN,2.42,2.56,10000
9,2026-04-26 18:00:00,28.7,32.08,28.7,28.7,70,light rain,100,0.61,NaN,1.71,2.01,10000


**3.2 Function for API calls and data preparation**

To call the weather API for each city, we are going to create a function that will request the API and save all the relevant information we have previously identified.

In [18]:
def fetch_weather_data(cities_df):
    """Fetch 7-day weather forecast for each city."""
    
    load_dotenv()
    API_KEY = os.getenv("API_KEY")
    
    if not API_KEY:
        raise ValueError("API_KEY not found in .env file")
    
    url_base = "https://api.openweathermap.org/data/2.5/forecast"
    results = {}
    
    for _, row in cities_df.iterrows():
        city = row['City']
        
        params = {
            "lat": row['Latitude'],
            "lon": row['Longitude'],
            "units": "metric",
            "appid": API_KEY
        }
        
        try:
            response = requests.get(url_base, params=params, timeout=10)
            
            if response.status_code != 200:
                results[city] = (pd.DataFrame(), False)
                continue
            
            # Parse and flatten JSON
            forecast_list = response.json()['list']
            df = pd.json_normalize(forecast_list, errors='ignore')
            
            # Extract weather description
            df['description'] = df['weather'].apply(
                lambda x: x[0]['description'] if isinstance(x, list) and len(x) > 0 else None
            )
            
            # Select and rename columns
            cols_to_keep = {
                'main.temp': 'temp',
                'main.temp_min': 'temp_min',
                'main.temp_max': 'temp_max',
                'main.humidity': 'humidity',
                'pop': 'rain_probability',
                'clouds.all': 'cloudiness',
                'wind.speed': 'wind_speed'
            }
            
            existing_cols = [col for col in cols_to_keep.keys() if col in df.columns]
            weather_df = df[['dt_txt', 'description'] + existing_cols].rename(columns=cols_to_keep)
            
            # Convert types
            weather_df['dt_txt'] = pd.to_datetime(weather_df['dt_txt'])
            for col in weather_df.columns:
                if col not in ['dt_txt', 'description']:
                    weather_df[col] = pd.to_numeric(weather_df[col], errors='coerce')        
            results[city] = (weather_df, True)
            
        except Exception as e:
            results[city] = (pd.DataFrame(), False)
    
    return results

**3.3 Weather API call for all cities**

Now we can use the created function to extract the weather information of the weather forecast for our list of 35 cities.

In [19]:
weather_all_cities_dict = fetch_weather_data(df_coord_35_cities)

In [20]:
weather_all_cities_dict

{'Mont Saint Michel': (                dt_txt       description   temp  temp_min  temp_max  humidity  \
  0  2026-04-25 15:00:00     broken clouds  19.06     18.73     19.06        58   
  1  2026-04-25 18:00:00  scattered clouds  17.36     16.43     17.36        71   
  2  2026-04-25 21:00:00        few clouds  12.12     12.12     12.12        86   
  3  2026-04-26 00:00:00  scattered clouds  10.14     10.14     10.14        77   
  4  2026-04-26 03:00:00   overcast clouds   9.32      9.32      9.32        80   
  5  2026-04-26 06:00:00     broken clouds   9.74      9.74      9.74        80   
  6  2026-04-26 09:00:00   overcast clouds  15.45     15.45     15.45        64   
  7  2026-04-26 12:00:00   overcast clouds  18.41     18.41     18.41        56   
  8  2026-04-26 15:00:00   overcast clouds  18.14     18.14     18.14        59   
  9  2026-04-26 18:00:00   overcast clouds  15.70     15.70     15.70        76   
  10 2026-04-26 21:00:00   overcast clouds  12.49     12.49     12

**3.3 Weather preferences**

For most people, a comfortable weather for holidays is : 
- hot but not too hot (around 25°C), 
- with low humidity, wind and clouds. 
- low rain probability

We are going to create a scoring functions with these parameters to determine the weather score of a given city. Then will be able to compare cities and select the best ones.

In [21]:
 
def score_city_weather(weather_df, ideal_temp=25):
    """Score a city's weather based on 7-day forecast."""
    
    # Aggregate by day
    daily = weather_df.groupby(weather_df['dt_txt'].dt.date).agg({
        'temp': 'mean',
        'temp_min': 'min',
        'temp_max': 'max',
        'humidity': 'mean',
        'rain_probability': 'mean'
    })
    
    # Calculate component scores [0, 1]
    temp_score = (1 - abs(daily['temp'] - ideal_temp) / 10).clip(0, 1)
    humidity_score = 1 - (daily['humidity'] / 100)
    rain_score = 1 - daily['rain_probability']
    stability_score = (1 - (daily['temp_max'] - daily['temp_min']) / 10).clip(0, 1)
    
    # Weighted final score
    daily_score = (
        0.4 * temp_score +
        0.2 * humidity_score +
        0.3 * rain_score +
        0.1 * stability_score
    )
    
    # City score: mean minus std (penalize variability)
    city_score = daily_score.mean() - daily_score.std()
    
    return {
        'avg_temp': daily['temp'].mean(),
        'avg_humidity': daily['humidity'].mean(),
        'rain_prob': daily['rain_probability'].mean() * 100,
        'temp_stability': daily['temp'].std(),
        'score': city_score
    }
 
 
def calculate_weather_scores(weather_results):
    """Calculate scores for all cities."""
    
    scores = []
    
    for city, (weather_df, success) in weather_results.items():
        if not success or weather_df.empty:
            continue
        
        stats = score_city_weather(weather_df)
        scores.append({
            'City': city,
            'Weather_Score': round(stats['score'], 3),
            'Avg_Temp': round(stats['avg_temp'], 1),
            'Avg_Humidity': round(stats['avg_humidity'], 1),
            'Rain_Probability': round(stats['rain_prob'], 1),
            'Temp_Stability': round(stats['temp_stability'], 2)
        })
    
    df = pd.DataFrame(scores).sort_values('Weather_Score', ascending=False).reset_index(drop=True)
    return df


**3.4. Weather scoring for all cities**

In [22]:
weather_score_all_df = calculate_weather_scores(weather_all_cities_dict)
weather_score_all_df

,City,Weather_Score,Avg_Temp,Avg_Humidity,Rain_Probability,Temp_Stability
0,Saintes Maries de la mer,0.435,16.8,72.8,2.8,0.76
1,Nimes,0.434,17.8,60.7,2.1,1.36
2,Carcassonne,0.425,18.2,72.3,7.6,1.42
3,Marseille,0.417,17.3,67.1,3.1,1.41
4,Toulouse,0.412,17.7,68.7,4.1,1.44
5,Aix en Provence,0.401,17.2,57.8,5.8,1.36
6,Avignon,0.396,17.5,60.5,1.9,1.66
7,Le Havre,0.396,12.8,66.5,0.0,1.29
8,Cassis,0.395,16.1,69.6,1.4,1.32
9,St Malo,0.389,4.8,49.1,0.0,2.11


**3.5. Export of the data**

In [23]:
weather_score_all_df.to_csv("DATA/bookinghotels/weather_best_destinations.csv", index=False)

Top five destinations according to our weather scoring :

In [24]:
top_five_destinations_df = weather_score_all_df.head(5).copy()
top_five_cities_names = weather_score_all_df.head(5)["City"].tolist()

In [25]:
print("The best cities according to the actual weather forecast are :\n")
for name, i in zip(top_five_cities_names,range(1,6)) : 
    print(f"  - N°{i} : {name}")

The best cities according to the actual weather forecast are :

  - N°1 : Saintes Maries de la mer
  - N°2 : Nimes
  - N°3 : Carcassonne
  - N°4 : Marseille
  - N°5 : Toulouse


In [26]:
top_five_destinations_df.to_csv("DATA/bookinghotels/top_five_destinations_csv.csv", index=True)

## **II. Identification of the 20 best hotels in the top-5 destinations**

Now that we have defined our top-5 list of destinations, let's check the 20 bests hotels per city using webscrapping techniques on the Booking.com website

### **1. Create a webscrapping function**

As Booking.com actively blocks scraping, we are using Selenium here to gather information for each hotel with : 
- its name,
- the url to its booking.com page,
- Its coordinates: latitude and longitude,
- Its score according to the website users,
- A text description of the hotel.

In [27]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
from datetime import datetime, timedelta, date

In [28]:
def scrape_hotels(city, check_in, check_out, num_hotels=20):
    """Scrape hotels for one city from Booking.com."""
    
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    options.add_argument("--no-sandbox")
    
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    
    hotels = []
    
    try:
        url = f"https://www.booking.com/searchresults.fr.html?ss={city}&checkin={check_in}&checkout={check_out}&nrpersons=2"
        driver.get(url)
        
        # Wait and scroll
        time.sleep(3)
        driver.execute_script("window.scrollBy(0, window.innerHeight);")
        time.sleep(2)
        
        # Extract hotels
        hotel_elements = driver.find_elements(By.CSS_SELECTOR, "div[data-testid='property-card']")
        
        for elem in hotel_elements[:num_hotels]:
            try:
                # Name
                name = elem.find_element(By.CSS_SELECTOR, "div[data-testid='title']").text
                
                # URL
                link = elem.find_element(By.CSS_SELECTOR, "a[href*='hotel']")
                url_hotel = link.get_attribute("href")
                if not url_hotel.startswith("http"):
                    url_hotel = "https://www.booking.com" + url_hotel
                
                # Rating
                rating = None
                try:
                    import re
                    rating_text = elem.text
                    match = re.search(r'(\d+[.,]\d)', rating_text)
                    if match:
                        rating = float(match.group(1).replace(',', '.'))
                except:
                    pass
                
                hotels.append({
                    'name': name,
                    'url': url_hotel,
                    'rating': rating,
                    'city': city
                })
            except:
                continue
    
    except Exception as e:
        print(f"Error scraping {city}: {str(e)}")
    
    finally:
        driver.quit()
    
    return hotels

### **2. Get hotels information**

In [29]:
def scrape_all_hotels(cities, check_in=None, check_out=None, num_hotels=20):
    """Scrape hotels for all cities."""
    
    if check_in is None:
        check_in = date.today()
        
    if check_out is None:
        check_out = check_in + timedelta(days=7)

    print(check_in, check_out)
    all_hotels = []
    
    for city in cities:
        hotels = scrape_hotels(city, check_in, check_out, num_hotels)
        all_hotels.extend(hotels)
    
    if not all_hotels:
        print("No hotels scraped!")
        return pd.DataFrame()
    
    df = pd.DataFrame(all_hotels)
    df = df.sort_values(['city', 'rating'], ascending=[True, False]).reset_index(drop=True)
    
    return df

In [30]:
cities = top_five_cities_names
assert isinstance(cities, list) and all(isinstance(c, str) for c in cities), \
    f"top_five_cities_names must be a string, received : {cities}"
print(f"Villes à scraper ({len(cities)}) : {cities}")

Villes à scraper (5) : ['Saintes Maries de la mer', 'Nimes', 'Carcassonne', 'Marseille', 'Toulouse']


In [31]:
hotels_df = scrape_all_hotels(cities)

2026-04-25 2026-05-02


In [32]:
hotels_df.head()

,name,url,rating,city
0,Le Petit Capoine 2- T2 Plein Centre de Carcass...,https://www.booking.com/hotel/fr/le-petit-capo...,9.8,Carcassonne
1,Le Clos Saint Michel,https://www.booking.com/hotel/fr/le-clos-saint...,9.2,Carcassonne
2,L'Impasse d'Adam Free Parking,https://www.booking.com/hotel/fr/impasse-d-ada...,9.1,Carcassonne
3,Le Casimir - Appartement de standing 90m2 - Ca...,https://www.booking.com/hotel/fr/le-casimir-h-...,8.8,Carcassonne
4,L'Eva-Ambre - 2 Chambres - Parking & Clim,https://www.booking.com/hotel/fr/eva-ambre-2-c...,8.7,Carcassonne


In [33]:
# Save to CSV
hotels_df.to_csv("DATA/bookinghotels/all_cities_hotels.csv", index=False)
hotels_df.head()


,name,url,rating,city
0,Le Petit Capoine 2- T2 Plein Centre de Carcass...,https://www.booking.com/hotel/fr/le-petit-capo...,9.8,Carcassonne
1,Le Clos Saint Michel,https://www.booking.com/hotel/fr/le-clos-saint...,9.2,Carcassonne
2,L'Impasse d'Adam Free Parking,https://www.booking.com/hotel/fr/impasse-d-ada...,9.1,Carcassonne
3,Le Casimir - Appartement de standing 90m2 - Ca...,https://www.booking.com/hotel/fr/le-casimir-h-...,8.8,Carcassonne
4,L'Eva-Ambre - 2 Chambres - Parking & Clim,https://www.booking.com/hotel/fr/eva-ambre-2-c...,8.7,Carcassonne


### **3. Visualizations**

**3.1. Top destinations map**

In [34]:
import plotly.express as px

def plot_top_destinations(df):
    """Interactive map with modern Mapbox-style background."""

    top5 = df.head(5)

    fig = px.scatter_mapbox(
        top5,
        lat="Latitude",
        lon="Longitude",
        text="City",
        zoom=4.5,
        center={"lat": 46.6, "lon": 2.5},  # France
        height=600
    )

    fig.update_traces(
        marker=dict(size=10, color="red"),
        hovertemplate="<b>%{text}</b><extra></extra>"
    )

    fig.update_layout(
        mapbox_style="carto-positron",  
        margin={"r":0,"t":40,"l":0,"b":0},
        title="<b>Top 5 Best Destinations in France</b>"
    )

    return fig

In [35]:
coordinates_35_cities = pd.read_csv("DATA/bookinghotels/coordinates_35_cities_csv.csv")

In [38]:
selected_cities = ['Saintes Maries de la mer',
                    'Nimes', 
                    'Carcassonne', 
                    'Marseille', 
                    'Toulouse']

# filtrer les 5 villes
top_cities_coordinates = coordinates_35_cities[coordinates_35_cities["City"].isin(selected_cities)]

In [39]:
top_cities_coordinates

,City,Latitude,Longitude
20,Marseille,43.296174,5.369953
24,Nimes,43.837425,4.360069
26,Saintes Maries de la mer,43.451592,4.427720
28,Carcassonne,43.213036,2.349107
30,Toulouse,43.604464,1.444243


In [40]:
plot_top_destinations(top_cities_coordinates)

**3.2. Top hotels map**

In [41]:
hotels_df.head()

,name,url,rating,city
0,Le Petit Capoine 2- T2 Plein Centre de Carcass...,https://www.booking.com/hotel/fr/le-petit-capo...,9.8,Carcassonne
1,Le Clos Saint Michel,https://www.booking.com/hotel/fr/le-clos-saint...,9.2,Carcassonne
2,L'Impasse d'Adam Free Parking,https://www.booking.com/hotel/fr/impasse-d-ada...,9.1,Carcassonne
3,Le Casimir - Appartement de standing 90m2 - Ca...,https://www.booking.com/hotel/fr/le-casimir-h-...,8.8,Carcassonne
4,L'Eva-Ambre - 2 Chambres - Parking & Clim,https://www.booking.com/hotel/fr/eva-ambre-2-c...,8.7,Carcassonne


As we cannot get the exact adress of the hotels from booking.com we will find them with the API Nominatum :

In [42]:
def geocode_hotels(hotels_df):
    url_base = "https://nominatim.openstreetmap.org/search"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/118.0"}
    latitudes, longitudes, addresses = [], [], []

    for _, row in hotels_df.iterrows():
        found = False
        queries = [
            f"{row['name']}, {row['city']}, France",
            f"{row['city']}, France"
        ]
        for query in queries:
            params = {"q": query, "format": "json", "addressdetails": "1", "limit": "1"}
            try:
                response = requests.get(url=url_base, headers=headers, params=params, timeout=10)
                time.sleep(1)  # Respect rate-limit 
                if response.status_code == 200 and response.json():
                    result = response.json()[0]
                    latitudes.append(float(result["lat"]))
                    longitudes.append(float(result["lon"]))
                    addresses.append(result.get("display_name", ""))
                    found = True
                    break
            except Exception:
                pass
        if not found:
            latitudes.append(None)
            longitudes.append(None)
            addresses.append("")

    enriched = hotels_df.copy()
    enriched["latitude"]  = latitudes
    enriched["longitude"] = longitudes
    enriched["address"]   = addresses
    return enriched

hotels_df = geocode_hotels(hotels_df)
hotels_df.to_csv("DATA/bookinghotels/all_cities_hotels_geocoded.csv", index=False)
print(f"Geocoded {hotels_df['latitude'].notna().sum()} / {len(hotels_df)} hotels")
hotels_df.head()

Geocoded 100 / 100 hotels


,name,url,rating,city,latitude,longitude,address
0,Le Petit Capoine 2- T2 Plein Centre de Carcass...,https://www.booking.com/hotel/fr/le-petit-capo...,9.8,Carcassonne,43.213036,2.349107,"Carcassonne, Aude, Occitanie, France métropoli..."
1,Le Clos Saint Michel,https://www.booking.com/hotel/fr/le-clos-saint...,9.2,Carcassonne,43.213036,2.349107,"Carcassonne, Aude, Occitanie, France métropoli..."
2,L'Impasse d'Adam Free Parking,https://www.booking.com/hotel/fr/impasse-d-ada...,9.1,Carcassonne,43.213036,2.349107,"Carcassonne, Aude, Occitanie, France métropoli..."
3,Le Casimir - Appartement de standing 90m2 - Ca...,https://www.booking.com/hotel/fr/le-casimir-h-...,8.8,Carcassonne,43.213036,2.349107,"Carcassonne, Aude, Occitanie, France métropoli..."
4,L'Eva-Ambre - 2 Chambres - Parking & Clim,https://www.booking.com/hotel/fr/eva-ambre-2-c...,8.7,Carcassonne,43.213036,2.349107,"Carcassonne, Aude, Occitanie, France métropoli..."


In [46]:
def plot_top_hotels(hotels_df, city=None):
    hotels_with_coords = hotels_df.dropna(subset=["latitude", "longitude"])
    if hotels_with_coords.empty:
        print("No hotels with coordinates. Run geocode_hotels() first.")
        return None
    if city:
        plot_df = (hotels_with_coords[hotels_with_coords["city"] == city]
                   .dropna(subset=["rating"])          
                   .nlargest(20, "rating"))
        title = f"<b>Top 20 Hotels — {city}</b>"
        zoom, center = 12, {"lat": plot_df["latitude"].mean(), "lon": plot_df["longitude"].mean()}
    else:
        plot_df = (hotels_with_coords
                   .dropna(subset=["rating"])         
                   .sort_values("rating", ascending=False)
                   .groupby("city").head(5).reset_index(drop=True))
        title = "<b>Top 5 Hotels per Destination</b>"
        zoom, center = 5, {"lat": 43.5, "lon": 4.0}

    if plot_df.empty:
        print(f"No hotels with valid rating{' in ' + city if city else ''}.")
        return None

    fig = px.scatter_mapbox(plot_df, lat="latitude", lon="longitude", color="city",
        size="rating", size_max=15, hover_name="name",
        hover_data={"rating": True, "city": True, "address": True,
                    "latitude": False, "longitude": False},
        zoom=zoom, center=center, height=600, title=title)
    fig.update_layout(mapbox_style="carto-positron", margin={"r":0,"t":40,"l":0,"b":0})
    return fig

In [47]:
fig_hotels_overview = plot_top_hotels(hotels_df)
if fig_hotels_overview:
    fig_hotels_overview.show()
for city_name in top_five_cities_names:
    fig_city = plot_top_hotels(hotels_df, city=city_name)
    if fig_city:
        fig_city.show()

### **4. Save elements to S3** ###

In [ ]:
! pip install boto3

In [55]:
import boto3
import logging
from dotenv import load_dotenv
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [56]:
load_dotenv()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Création du client S3 avec les credentials du .env
s3_client = boto3.client(
    's3',
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_DEFAULT_REGION")
)

In [57]:
def upload_to_s3(dataframe, filename, bucket_name):
    try:
        csv_buffer = dataframe.to_csv(index=False).encode('utf-8')
        s3_client.put_object(Bucket=bucket_name, Key=filename, Body=csv_buffer)
        logger.info(f"✓ Uploaded {filename} to s3://{bucket_name}/")
    except Exception as e:
        logger.error(f"Failed to upload {filename}: {str(e)}")

def save_figure_to_s3(figure, filename, bucket_name):
    try:
        html_string = figure.to_html()
        s3_client.put_object(
            Bucket=bucket_name,
            Key=filename,
            Body=html_string.encode('utf-8'),
            ContentType='text/html'
        )
        logger.info(f"✓ Uploaded {filename} to s3://{bucket_name}/")
    except Exception as e:
        logger.error(f"Failed to upload {filename}: {str(e)}")

In [59]:
BUCKET = "kayak-project-mt"
PREFIX = "weather and hotels/"

upload_to_s3(df_coord_35_cities,       PREFIX + "data/coordinates_35_cities.csv",     BUCKET)
upload_to_s3(weather_score_all_df,     PREFIX + "data/weather_best_destinations.csv",  BUCKET)
upload_to_s3(top_five_destinations_df, PREFIX + "data/top_five_destinations.csv",      BUCKET)
upload_to_s3(hotels_df,                PREFIX + "data/all_cities_hotels_geocoded.csv", BUCKET)

fig_destinations = plot_top_destinations(top_cities_coordinates)
save_figure_to_s3(fig_destinations,    PREFIX + "maps/top_destinations_map.html",      BUCKET)
save_figure_to_s3(fig_hotels_overview, PREFIX + "maps/top_hotels_overview.html",       BUCKET)
for city_name in top_five_cities_names:
    fig_city = plot_top_hotels(hotels_df, city=city_name)
    if fig_city:
        city_slug = city_name.lower().replace(" ", "_")
        save_figure_to_s3(fig_city, PREFIX + f"maps/hotels_{city_slug}.html", BUCKET)

INFO:__main__:✓ Uploaded weather and hotels/data/coordinates_35_cities.csv to s3://kayak-project-mt/
INFO:__main__:✓ Uploaded weather and hotels/data/weather_best_destinations.csv to s3://kayak-project-mt/
INFO:__main__:✓ Uploaded weather and hotels/data/top_five_destinations.csv to s3://kayak-project-mt/
INFO:__main__:✓ Uploaded weather and hotels/data/all_cities_hotels_geocoded.csv to s3://kayak-project-mt/
INFO:__main__:✓ Uploaded weather and hotels/maps/top_destinations_map.html to s3://kayak-project-mt/
INFO:__main__:✓ Uploaded weather and hotels/maps/top_hotels_overview.html to s3://kayak-project-mt/
INFO:__main__:✓ Uploaded weather and hotels/maps/hotels_saintes_maries_de_la_mer.html to s3://kayak-project-mt/
INFO:__main__:✓ Uploaded weather and hotels/maps/hotels_nimes.html to s3://kayak-project-mt/
INFO:__main__:✓ Uploaded weather and hotels/maps/hotels_carcassonne.html to s3://kayak-project-mt/
INFO:__main__:✓ Uploaded weather and hotels/maps/hotels_marseille.html to s3://kay

## **Conclusion**

This project combined weather forecasting APIs, geolocation data, and web scraping to recommend the best holiday destinations in France for the coming week.

Based on the 7-day weather forecast scored across temperature, humidity, rainfall probability and stability, the **top 5 destinations** identified are:

1. **Saintes-Maries-de-la-Mer**
2. **Nîmes**
3. **Carcassonne**
4. **Marseille**
5. **Toulouse**

All five cities are located in the south of France, which is consistent with the expected sunny and dry weather conditions typical of the Mediterranean and Occitanie regions in spring.

For each of these destinations, the top-rated hotels were scraped from Booking.com and geocoded using the Nominatim API, then visualised on interactive maps.

### *Limitations*

- **Booking.com actively blocks automated scraping**, which means the number of hotels retrieved per city is limited and may not be representative of all available options. Some hotels may be missing from the maps entirely due to failed scraping attempts or bot-detection mechanisms.
- The weather scoring model is based on a **fixed set of preferences** (ideal temperature ~25°C, low humidity and rain probability). Results would differ for users with different comfort thresholds.
- The Nominatim API has a **rate limit of 1 request per second**, which slows down geocoding and can occasionally return approximate coordinates — especially for hotel names that are not well-referenced in OpenStreetMap.
- The analysis is a **snapshot in time**: re-running the notebook a few days later would likely produce different top destinations as the weather forecast updates.